### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import glob
sys.path.append(os.path.abspath('..'))

from quinex_application.scopus_data import ProcessScopusData
from quinex_application.quinex_postprocessing import QuinexPostprocessing

In [ ]:
output_path = "../data/paper/paper_dataset.parquet"
csvs_paths = glob.glob("../data/paper/wos_query4/by_years/wos_*.xls")
query_name = 'Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))'
query_date = "10.06.2025"
quelle_name = "Source: WOS"

scopus = ProcessScopusData(csvs_paths, output_path, query_name, query_date, quelle_name) 
qpost = QuinexPostprocessing()

## **Load and Clean CSV Files**

### **Combine the CSV Files**

In [ ]:
df = scopus.combine_csvs()
print(len(df))

### **Clean duplicate DOIs**

In [ ]:
duplicate_doi_values = df["DOI"].value_counts()
duplicate_doi_values = duplicate_doi_values[duplicate_doi_values > 1]
print(f"Anzahl verschiedener mehrfacher DOIs: {len(duplicate_doi_values)}")
print(f"Anzahl an Zeilen mit mehrfacher DOI: {duplicate_doi_values.sum()}")

In [ ]:
df_cleaned = scopus.remove_duplicates_doi()
print(len(df_cleaned))

### **Add new Papers to Parquet**

In [ ]:
# TODO: Funktion weiter testen
parquet = scopus.update_parquet_with_new_data()
display(parquet)

### **Save cleaned and added data**

In [ ]:
df = scopus.save_df()

## **Select specific Query**

In [ ]:
df = qpost.load_df("../data/paper/paper_dataset.parquet")

In [ ]:
df_scopus_failed = df[(df["Download_Status"] == "Failed") & (df['Source: WOS'] == True)]
df_scopus_failed = df_scopus_failed[df_scopus_failed["Year"] >= 2010].drop_duplicates(subset="DOI")
len(df_scopus_failed)

In [ ]:
import pandas as pd

print(len(df_scopus_failed))
# NaN und Duplikate entfernen
df_scopus_failed = df_scopus_failed.dropna(subset=['DOI'])
print(len(df_scopus_failed))


dois = df_scopus_failed['DOI'].astype(str).str.strip().tolist()

query = " OR ".join([f'DO=({d})' for d in dois])

# Datei speichern
filename = f"../data/paper/wos_query4_abstracts/wos_query.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(query)

print(f"✅ Query mit {len(dois)} DOIs gespeichert ({filename})")

In [ ]:
path = "../data/paper/wos_query4_abstracts/wos.xls"
df_2025 = pd.read_excel(path)
display(df_2025)
# df_2025["abstract_length"] = df_2025["Abstract"].astype(str).str.len()

# # Pro DOI den längsten Abstract auswählen
# df_2025_unique = (
#     df_2025.sort_values("abstract_length", ascending=False)
#            .drop_duplicates(subset="DOI", keep="first")
#            .drop(columns="abstract_length")
# )
# print(f"✅ Von {len(df_2025)} Einträgen auf {len(df_2025_unique)} eindeutige DOIs reduziert.")
# df_2025_unique.to_csv(path, index=False, encoding="utf-8")

In [ ]:
path = "../data/paper/scopus_query4_abstracts/by_year/scopus_2010.csv"
df = pd.read_csv(path)
df_modified = df[df['Abstract'] != "[No abstract available]"]
print(f"{len(df)}/{len(df_modified)}")
df_modified.to_csv(path, index=False, encoding="utf-8")